In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!nvidia-smi

Tue May  5 04:42:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
    

CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Test with a Hindi prompt
messages = [
    {"role": "user", "content": "मुझे भारत के बारे में 3 रोचक तथ्य बताओ"}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_token=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
मुझे भारत के बारे में 3 रोचक तथ्य बताओ
system
1. **भारत सरकार की श्रृंखला:** भारत की सरकार की पहली आयोजनियों के लिए देखभाल किया गया है। इसके बाद, विश्वविद्यालय और उप-विद्यालय के अधिकारी ने फ्रांसीसी चाँदनी जैसे टीकेस नियंत्रण देने का एक आयोजन किया।

2. **भारत की आर्थिक सटीकता:** भारत ने नियमित रूप से विश्व स्थापना के अनुसा�


In [5]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
WANDB_KEY = secrets.get_secret("WANDB_API_KEY")

# Login to HuggingFace
from huggingface_hub import login
login(token=HF_TOKEN)

# Login to W&B
import wandb
wandb.login(key=WANDB_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: naveenrumawat756 (naveenrumawat756-the-lnm-institute-of-infromation-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [20]:
from transformers import AutoTokenizer

# Qwen (your model) vs Google Gemma (ungated, similar size)
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
gemma_tok = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")

test_sentences = [
    "भारत एक विशाल देश है जिसमें अनेक भाषाएँ बोली जाती हैं",
    "मुझे आज का मौसम बताओ",
    "किसान को फसल बोने से पहले मिट्टी की जांच करनी चाहिए",
    "yaar mujhe ek accha restaurant bata do nearby",
]

for sent in test_sentences:
    q_tokens = qwen_tok.tokenize(sent)
    g_tokens = gemma_tok.tokenize(sent)
    print(f"\nText: {sent}")
    print(f"Qwen  tokens ({len(q_tokens)}): {q_tokens[:8]}")
    print(f"Gemma tokens ({len(g_tokens)}): {g_tokens[:8]}")
    print(f"Ratio: Qwen uses {len(q_tokens)/len(g_tokens):.2f}x tokens vs Gemma")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]


Text: भारत एक विशाल देश है जिसमें अनेक भाषाएँ बोली जाती हैं
Qwen  tokens (54): ['à¤Ń', 'à¤¾à¤', '°', 'à¤¤', 'Ġà¤', 'ı', 'à¤ķ', 'Ġà¤']
Gemma tokens (17): ['भार', 'त', '▁एक', '▁विश', 'ाल', '▁देश', '▁है', '▁जिसमें']
Ratio: Qwen uses 3.18x tokens vs Gemma

Text: मुझे आज का मौसम बताओ
Qwen  tokens (19): ['à¤®', 'à¥ģ', 'à¤Ŀ', 'à¥ĩ', 'Ġà¤', 'Ĩ', 'à¤ľ', 'Ġà¤ķ']
Gemma tokens (8): ['मु', 'झे', '▁आज', '▁का', '▁मौ', 'सम', '▁बता', 'ओ']
Ratio: Qwen uses 2.38x tokens vs Gemma

Text: किसान को फसल बोने से पहले मिट्टी की जांच करनी चाहिए
Qwen  tokens (45): ['à¤ķ', 'à¤¿à¤', '¸', 'à¤¾à¤', '¨', 'Ġà¤ķ', 'à¥ĭ', 'Ġà¤']
Gemma tokens (20): ['क', 'िस', 'ान', '▁को', '▁फ', 'स', 'ल', '▁बो']
Ratio: Qwen uses 2.25x tokens vs Gemma

Text: yaar mujhe ek accha restaurant bata do nearby
Qwen  tokens (12): ['ya', 'ar', 'Ġmuj', 'he', 'Ġek', 'Ġac', 'cha', 'Ġrestaurant']
Gemma tokens (11): ['yaar', '▁mu', 'j', 'he', '▁ek', '▁acc', 'ha', '▁restaurant']
Ratio: Qwen uses 1.09x tokens vs Gemma


In [3]:
from transformers import AutoTokenizer

qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
gemma_tok = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
llama_tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

test_sentences = [
    "भारत एक विशाल देश है जिसमें अनेक भाषाएँ बोली जाती हैं",
    "मुझे आज का मौसम बताओ",
    "किसान को फसल बोने से पहले मिट्टी की जांच करनी चाहिए",
    "yaar mujhe ek accha restaurant bata do nearby",
]

for sent in test_sentences:
    q = qwen_tok.tokenize(sent)
    g = gemma_tok.tokenize(sent)
    l = llama_tok.tokenize(sent)
    print(f"\nText: {sent}")
    print(f"Qwen  ({len(q):2d} tokens): {q[:6]}")
    print(f"Gemma ({len(g):2d} tokens): {g[:6]}")
    print(f"Llama ({len(l):2d} tokens): {l[:6]}")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]


Text: भारत एक विशाल देश है जिसमें अनेक भाषाएँ बोली जाती हैं
Qwen  (54 tokens): ['à¤Ń', 'à¤¾à¤', '°', 'à¤¤', 'Ġà¤', 'ı']
Gemma (17 tokens): ['भार', 'त', '▁एक', '▁विश', 'ाल', '▁देश']
Llama (27 tokens): ['à¤Ń', 'à¤¾à¤°à¤¤', 'Ġà¤ıà¤ķ', 'Ġà¤µ', 'à¤¿à¤¶', 'à¤¾à¤²']

Text: मुझे आज का मौसम बताओ
Qwen  (19 tokens): ['à¤®', 'à¥ģ', 'à¤Ŀ', 'à¥ĩ', 'Ġà¤', 'Ĩ']
Gemma ( 8 tokens): ['मु', 'झे', '▁आज', '▁का', '▁मौ', 'सम']
Llama (11 tokens): ['à¤®', 'à¥ģà¤Ŀ', 'à¥ĩ', 'Ġà¤Ĩà¤ľ', 'Ġà¤ķ', 'à¤¾']

Text: किसान को फसल बोने से पहले मिट्टी की जांच करनी चाहिए
Qwen  (45 tokens): ['à¤ķ', 'à¤¿à¤', '¸', 'à¤¾à¤', '¨', 'Ġà¤ķ']
Gemma (20 tokens): ['क', 'िस', 'ान', '▁को', '▁फ', 'स']
Llama (28 tokens): ['à¤ķ', 'à¤¿à¤¸', 'à¤¾à¤¨', 'Ġà¤ķ', 'à¥ĭ', 'Ġà¤«']

Text: yaar mujhe ek accha restaurant bata do nearby
Qwen  (12 tokens): ['ya', 'ar', 'Ġmuj', 'he', 'Ġek', 'Ġac']
Gemma (11 tokens): ['yaar', '▁mu', 'j', 'he', '▁ek', '▁acc']
Llama (12 tokens): ['ya', 'ar', 'Ġmuj', 'he', 'Ġek', 'Ġac']


In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

prompt = """तुम एक सहायक AI हो।
नीचे दिए गए प्रश्न का उत्तर हिंदी में दो।

प्रश्न: मुझे भारत के बारे में 3 रोचक तथ्य बताओ
उत्तर:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


तुम एक सहायक AI हो।
नीचे दिए गए प्रश्न का उत्तर हिंदी में दो।

प्रश्न: मुझे भारत के बारे में 3 रोचक तथ्य बताओ
उत्तर: 

1. **भारत दुनिया का सबसे बड़ा उपनिवेश है।** भारत दुनिया का सबसे बड़ा उपनिवेश है, जिसका अर्थ है कि यह दुनिया में सबसे अधिक लोगो का घर है। 
2. **भारत दुनिया का सबसे बड़ा बाजार है।** भारत दुनिया का सबसे बड़ा बाजार है, जिसका अर्थ है कि यह दुनिया में सबसे अधिक सामान बेचता है। 
3. **भारत दुनिया का सबसे बड़ा कृषि उत्पादक है।** भारत दुनिया का सबसे बड़ा कृषि उत्पादक है, जिसका अर्थ है कि यह दुनिया में सबसे अधिक खाद्य पदार्थों का उत्पादन करता है। 


**


In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

messages = [
    {"role": "user", "content": "मुझे भारत के बारे में 3 रोचक तथ्य बताओ"}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_token=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

user
मुझे भारत के बारे में 3 रोचक तथ्य बताओ
* 
* 
* 

Here are three interesting facts about India:

1. **India is home to the world's largest democracy.**  With over 1.3 billion people, India has a vibrant democracy with a long history of peaceful transitions of power. 
2. **India is a land of ancient history and culture.**  From the Indus Valley Civilization to the Mughal Empire, India boasts a rich tapestry of ancient civilizations and empires. 
3. **India is a land of incredible biodiversity.**  With a diverse range of ecosystems, from the Himalayas to the tropical rainforests, India is home to a vast array of flora and fauna. 


Let me know if you'd like to know more! 

